<a href="https://colab.research.google.com/github/robsongfk/SalesInsightPY/blob/main/salesinsight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# =====================================================================
# RF01 - Criar ou Carregar o Dataset de Vendas (Código exato do PDF)
# =====================================================================
def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor", "Teclado", "Mouse", "Headset"]
    categorias = {
        "Notebook": "Computadores", "Smartphone": "Celulares",
        "Tablet": "Celulares", "Monitor": "Computadores",
        "Teclado": "Perifericos", "Mouse": "Perifericos", "Headset": "Perifericos"
    }
    precos = {
        "Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
        "Monitor": 1200, "Teclado": 250, "Mouse": 120, "Headset": 350
    }
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    data_inicio = datetime(2025, 1, 1)

    dados = []
    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza
        if random.random() < 0.05: quantidade = None # valor nulo
        if random.random() < 0.04: preco = None # valor nulo
        if random.random() < 0.06: produto = " " + produto + " " # espacos extras
        if random.random() < 0.03: data_txt = "DATA INVALIDA" # data invalida
        if random.random() < 0.10: # ruido no nome
            cliente = random.choice([
                cliente.upper().replace("_", "-"),
                cliente + "!!",
                " " + cliente,
                cliente.replace("Cliente_", "cliente#"),
            ])

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })
    return pd.DataFrame(dados)

# Gerar e salvar o CSV bruto
if not os.path.exists("vendas.csv"):
    df_bruto = gerar_dataset_vendas()
    df_bruto.to_csv("vendas.csv", index=False)
    print(f"Dataset gerado com {len(df_bruto)} registros.\n")


# =====================================================================
# RF02 - Inspecionar e Descrever os Dados
# =====================================================================
def inspecionar_dados(df):
    """Exibe as informacoes estruturais do DataFrame."""
    print("=== INSPECAO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df

# Testando as etapas 1 e 2
df_carregado = pd.read_csv("vendas.csv")
inspecionar_dados(df_carregado)

=== INSPECAO INICIAL DO DATASET ===
Shape: (365, 6)

Colunas: ['ID_Transacao', 'Data', 'Cliente', 'Categoria', 'Quantidade', 'Preco_Unitario']

Tipos de dados:
ID_Transacao        int64
Data               object
Cliente            object
Categoria          object
Quantidade        float64
Preco_Unitario    float64
dtype: object

Valores nulos por coluna:
ID_Transacao      0
Data              0
Cliente           0
Categoria         0
Quantidade        6
Preco_Unitario    0
dtype: int64

Primeiros registros:
   ID_Transacao        Data     Cliente      Categoria  Quantidade  \
0          1001  2023-04-13   Cliente_2   Eletrônicos          4.0   
1          1002  2023-12-15  Cliente_27   Eletrônicos          9.0   
2          1003  2023-09-28  Cliente_42         Roupas         9.0   
3          1004  2023-04-17   Cliente_2   Casa e Banho         2.0   
4          1005  2023-03-13  Cliente_26   Eletrônicos          8.0   

   Preco_Unitario  
0      226.105759  
1      208.197870  
2      

,ID_Transacao,Data,Cliente,Categoria,Quantidade,Preco_Unitario
0,1001,2023-04-13,Cliente_2,Eletrônicos,4.0,226.105759
1,1002,2023-12-15,Cliente_27,Eletrônicos,9.0,208.197870
2,1003,2023-09-28,Cliente_42,Roupas,9.0,290.489602
3,1004,2023-04-17,Cliente_2,Casa e Banho,2.0,403.948720
4,1005,2023-03-13,Cliente_26,Eletrônicos,8.0,273.816730
...,...,...,...,...,...,...
360,1361,2023-10-07,Cliente_9,Eletrônicos,3.0,230.222579
361,1362,2023-04-24,Cliente_33,Eletrônicos,2.0,298.063342
362,1363,2023-10-15,Cliente_20,Casa e Banho,3.0,474.805844
363,1364,2023-12-08,Cliente_13,Casa e Banho,5.0,133.738613
